# Laboratorio 4 Búsqueda en entornos complejos 
Sharis Barrios 221370 <br>
Emilio Reyes 22674

In [1]:
# librerias 
import random
import time

In [3]:
def es_solucion(estado):
    """
    Verifica si un estado representa una solución válida al problema
    de las 8 reinas.

    Parámetros:
        estado (list): lista de 8 enteros, donde el índice representa
                       la columna y el valor representa la fila.

    Retorna:
        bool: True si es una solución válida, False en caso contrario.
    """
    if not isinstance(estado, list) or len(estado) != 8:
        return False

    # Verificar que todas las filas sean enteros entre 0 y 7
    for fila in estado:
        if not isinstance(fila, int) or fila < 0 or fila > 7:
            return False

    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            # Misma fila
            if r1 == r2:
                return False

            # Misma diagonal
            if abs(r1 - r2) == abs(c1 - c2):
                return False

    return True

print(es_solucion([0, 4, 7, 5, 2, 6, 1, 3]))  # True
print(es_solucion([0, 1, 2, 3, 4, 5, 6, 7]))  # False


def heuristica(estado):
    """
    Calcula el número de pares de reinas que se atacan entre sí.
    Un estado solución tiene heurística 0.
    """
    conflictos = 0
    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            if r1 == r2 or abs(r1 - r2) == abs(c1 - c2):
                conflictos += 1

    return conflictos

True
False


In [4]:
# Generar estado aleatorio 
def estado_aleatorio():
    return [random.randint(0,7) for _ in range(8)]

In [5]:
# Generar vecinos 
def generar_vecinos(estado):

    vecinos = []

    for columna in range(8):

        fila_actual = estado[columna]

        for nueva_fila in range(8):

            if nueva_fila != fila_actual:

                nuevo_estado = estado.copy()
                nuevo_estado[columna] = nueva_fila

                vecinos.append(nuevo_estado)

    return vecinos

## Hill Climbing 

In [6]:
def hill_climbing():
    estado = estado_aleatorio()
    pasos = 0

    while True:
        h_actual = heuristica(estado)

        # Si ya es solución
        if h_actual == 0:
            return True, pasos, h_actual

        # Generar vecinos
        vecinos = generar_vecinos(estado)

        # Elegir el mejor vecino
        mejor_vecino = min(vecinos, key=heuristica)
        h_mejor = heuristica(mejor_vecino)

        # Si no mejora, nos quedamos atrapados
        if h_mejor >= h_actual:
            return False, pasos, h_actual

        # Moverse al mejor vecino
        estado = mejor_vecino
        pasos += 1

In [7]:
for _ in range(20):
    print(hill_climbing())

#imprime si llega a la solución, pasos y heuristica final

(False, 4, 1)
(False, 3, 2)
(False, 2, 3)
(False, 2, 2)
(False, 3, 1)
(False, 4, 1)
(False, 2, 3)
(False, 5, 1)
(False, 4, 1)
(False, 4, 1)
(False, 2, 1)
(True, 5, 0)
(False, 3, 1)
(False, 5, 1)
(False, 2, 1)
(False, 3, 1)
(False, 2, 2)
(True, 5, 0)
(False, 3, 1)
(False, 4, 1)


## Hill Climbing (Random Restart)

In [8]:
def random_restart(max_restarts=100):
    total_pasos = 0

    for _ in range(max_restarts):
        exito, pasos, h = hill_climbing()
        total_pasos += pasos

        if exito:
            return True, total_pasos, 0

    # Si nunca encontró solución
    return False, total_pasos, h

In [9]:
for _ in range(20):
    print(random_restart())

(True, 12, 0)
(True, 37, 0)
(True, 42, 0)
(True, 5, 0)
(True, 12, 0)
(True, 23, 0)
(True, 8, 0)
(True, 4, 0)
(True, 17, 0)
(True, 33, 0)
(True, 6, 0)
(True, 11, 0)
(True, 54, 0)
(True, 11, 0)
(True, 19, 0)
(True, 7, 0)
(True, 12, 0)
(True, 23, 0)
(True, 3, 0)
(True, 64, 0)


## Beam Search

In [10]:
def beam_search(k=3, max_iter=100):
    # k estados iniciales
    estados = [estado_aleatorio() for _ in range(k)]
    pasos = 0

    for _ in range(max_iter):
        # Revisar si ya hay solución
        for estado in estados:
            if heuristica(estado) == 0:
                return True, pasos, 0

        # Generar todos los vecinos de todos los estados
        todos_vecinos = []
        for estado in estados:
            vecinos = generar_vecinos(estado)
            todos_vecinos.extend(vecinos)

        # Elegir los k mejores vecinos
        todos_vecinos.sort(key=heuristica)
        estados = todos_vecinos[:k]

        pasos += 1

    # Si no encontró solución
    mejor_h = min(heuristica(e) for e in estados)
    return False, pasos, mejor_h

In [11]:
for _ in range(20):
    print(beam_search(k=5))

(False, 100, 1)
(True, 5, 0)
(False, 100, 1)
(True, 3, 0)
(True, 3, 0)
(False, 100, 1)
(True, 3, 0)
(False, 100, 1)
(True, 5, 0)
(True, 5, 0)
(True, 4, 0)
(False, 100, 1)
(False, 100, 1)
(False, 100, 1)
(True, 3, 0)
(True, 5, 0)
(True, 3, 0)
(True, 3, 0)
(True, 6, 0)
(True, 5, 0)


## Experimentos

In [14]:
def experimento(algoritmo, n=1000):
    exitos = 0
    pasos_total = 0
    heuristica_total = 0

    inicio = time.time()  # inicio del tiempo total

    for _ in range(n):
        exito, pasos, h = algoritmo()

        if exito:
            exitos += 1

        pasos_total += pasos
        heuristica_total += h

    fin = time.time()  # fin del tiempo total

    # Calculo de tiempo de ejecución
    tiempo_total = fin - inicio
    tiempo_promedio = tiempo_total / n

    return {
        "porcentaje_exito": exitos / n,
        "pasos_promedio": pasos_total / n,
        "heuristica_promedio": heuristica_total / n,
        "tiempo_promedio": tiempo_promedio
    }

In [18]:
def beam_k():
    return beam_search(k=5)

def beam_k1():
    return beam_search(k=2)

def beam_k2():
    return beam_search(k=10)

In [19]:
print("Hill Climbing:")
print(experimento(hill_climbing))

print("\nRandom Restart:")
print(experimento(random_restart))

print("\nBeam Search (k=5):")
print(experimento(beam_k))

print("\nBeam Search (k=2):")
print(experimento(beam_k1))

print("\nBeam Search (k=10):")
print(experimento(beam_k2))

Hill Climbing:
{'porcentaje_exito': 0.124, 'pasos_promedio': 3.204, 'heuristica_promedio': 1.267, 'tiempo_promedio': 0.0008134443759918213}

Random Restart:
{'porcentaje_exito': 1.0, 'pasos_promedio': 21.223, 'heuristica_promedio': 0.0, 'tiempo_promedio': 0.00569292163848877}

Beam Search (k=5):
{'porcentaje_exito': 0.703, 'pasos_promedio': 32.685, 'heuristica_promedio': 0.302, 'tiempo_promedio': 0.035871402740478515}

Beam Search (k=2):
{'porcentaje_exito': 0.499, 'pasos_promedio': 52.372, 'heuristica_promedio': 0.572, 'tiempo_promedio': 0.03027629017829895}

Beam Search (k=10):
{'porcentaje_exito': 0.872, 'pasos_promedio': 16.173, 'heuristica_promedio': 0.129, 'tiempo_promedio': 0.045190638542175296}


# Discusión 

**¿Qué tan frecuentemente Hill Climbing encuentra una solución?** 

El algoritmo presentó una tasa de éxito relativrealmenteamente baja, alcanzando un 12% de soluciones exitosas. Esto también se evidencia en las corridas realizadas, donde solo 1 de 20 veces se obtuvo una solución al problema. Esto indica que el algoritmo no logra encontrar una configuración válida del problema de las 8 reinas. Aunque su tiempo promedio de ejecución es bastante bajo (0.0008 segundos) y requiere pocos pasos en promedio, su capacidad para resolver el problema es limitada. Por lo cual, no es un algoritmo útil para encontrar una solución de manera satisfactoria. 


**¿Qué tipo de problemas presenta Hill Climbing en este dominio?**

El problema más grande de este algoritmo radica en los óptimos locales, mesetas y crestas. En el contexto de nuestro problema de las 8 reinas, el algoritmo tiende a quedarse atrapado en estados donde no existe un vecino cercano que mejore la heurística (aunque si exista una solución global). Esto queda evidenciado en los pasos promedio del algoritmo, ya que se detiene rápidamente y no alcanza la solución óptima. Así también, manteniendo un valor promedio de heurística mayor a cero (1.267). 


**¿La variante elegida mejora el desempeño? ¿Por qué?**

Definitivamente. La variante de Random Restart Hill Climbing mejora totalmente el desempeño de búsqueda de una solución. En este caso, se obtuvo un 100% de éxito lo que indica que el algoritmo siempre encuentra una solución. Esto sucede porque al reiniciar el algoritmo varias veces con diferentes estados iniciales, evitamos quedar atrapado en óptimos locales. Y aunque el número de pasos promedio aumenta significativamente (de 23.204 a 22.24) y el tiempo de ejecución es mayor (0.006 segundos) la mejora en la tasa de éxito justifica estos costos adicionales.


**¿Cómo afecta el valor de k en Beam Search?** 

En Beam Search el parámetro k (beam width) controla cuántos estados se mantienen en cada nivel de búsqueda. En este caso, con un valor de k = 5, se obtuvo una tasa de éxito del 70.3%, lo cual representa un equilibrio entre exploración y eficiencia. A mayor valor de k, el algoritmo puede explorar más alternativas y aumenta la probabilidad de encontrar una solución. Tal es el caso cuando tenemos k = 10 donde la tasa de éxito sube a un 87.2%. Pero nótese que el costo computacional del algoritmo es mayor al Hill Climbing y Random Restart Hill Climbing, y aumenta conforme K es mayor (K = 2 - 0.0302 segundos, K = 5 - 0.035 segundos y K = 10 - 0.045 segundos). 

**¿Cuál algoritmo resultó más efectivo?**

El algoritmo más efectivo fue Random Restart Hill Climbing, ya que logró un 100% de éxito y una heurística final promedio de 0.0. Esto nos indica que el algoritmo SIEMPRE encuentra soluciones válidas. Nótese también que Beam Search mostró buen desempeño (70% y 87% de éxito con K = 5 y K = 10 respectivamente), pero no alcanza la consistencia del Random Restart y su costo computacional es mucho más alto; por lo cual su eficiencia y desempeño no son comparables. 


**¿Qué relación existe entre costo computacional y tasa de éxito?**

Se observa una relación directa entre el costo computacional y la tasa de éxito para: Hill Climbing que es el algoritmo más rápido, pero tiene la menor tasa de éxito. Por el contrario, Random Restart incrementa el tiempo de ejecución y el número de pasos, pero logra la mayor tasa de éxito excepcional. 

Por su parte, el algoritmo Beam Search evidencia también una relación directa positiva entre el costo computacional y la tasa de éxitsino. Sin embargo, nótese que la relación es más clara al comparar distintos valores de k dentro del algoritmo,que al comparar sus resultados con los algoritmos de Hill Climbing o Random Restart.

Entonces, a medida que el valor de k aumenta (de k=2 a k=5 y k=10), se incrementa la tasa de éxito, pasando de 49.9% a 70.3% y hasta 87.2% respectivamente. Este aumento se acompaña de un mayor tiempo promedio de ejecución, debido a la exploración de más estados. Así también, se observa que el número de pasos promedio de Beam Search a diferencia de los algoritmos anteriores, disminuye conforme aumenta k (de 52.372 en k=2 a 32.685 en k=5 y 16.173 en k=10). Esto indica que al explorar más alternativas en cada iteración, el algoritmo logra encontrar soluciones de manera más directa. 